# Explainable Physics-Guided Deep Learning with SAR-Wake Geometry Fusion
## Complete End-to-End Master Pipeline (Wake-Net + Radon + MMDEC Full Harvest + ConvNeXt/Lee/Geometry Ensemble)

Executes the complete Master Plan directly on Kaggle GPU:
1. **Stage 1 (UEIKAP Wake-Net):** ResNet-50 + scSE Attention U-Net trained on human-annotated UEIKAP masks + Radon Transform kinematic validation.
2. **Stage 2 (MMDEC High-Yield Harvester):** Downloads `Dataset_AIS_POS.parquet` (430 MB) + `Dataset_S1_chunks.zip` (10.5 GB, streamed directly from zip with zero extra disk extraction!), computes exact unwrapped `d(COG)/dt` Rate-of-Turn, harvests ALL available turning vessels (Port & Starboard) + up to 4,000+ straight cruising vessels (configurable `--max_straight -1` for all), and runs Wake-Net + Radon auto-masking.
3. **Stage 3 (Method 1 + Method 3 Dual Ensemble):** 7x7 Adaptive Lee Filter + ConvNeXt-Tiny + Geometry Specialist MLP Soft Voting with Physics-Informed Multi-Task Loss.
4. **Stage 4 (Multi-Task Evaluation):** Binary Maneuver State, 3-Class Directional (Straight vs Port vs Starboard), Continuous Signed ROT (deg/min), Operational Speed (FPS/ms), and Before/After Lee Filter Grad-CAM.
5. **Stage 5 (Export):** Packages weights, metrics, predictions, logs, and Grad-CAM visuals into `/kaggle/working/master_thesis_results_3k.zip`.

In [ ]:
import os
import shutil

print("=== 1. CLONING MASTER THESIS REPOSITORY ===")
if os.path.exists('/kaggle/working/thesis/.git'):
    !git -C /kaggle/working/thesis fetch origin && !git -C /kaggle/working/thesis reset --hard origin/main
else:
    !git clone https://github.com/mohaimin2103001/thesis.git /kaggle/working/thesis
%cd /kaggle/working/thesis

print("\n=== 2. INSTALLING DEPENDENCIES ===")
!pip install -q pyarrow scikit-image shapely pyyaml timm albumentations segmentation-models-pytorch onnx onnxruntime pypdf tqdm

import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
print("=== 3. DOWNLOADING UEIKAP + MMDEC DATASETS IN KAGGLE CLOUD ===")
!mkdir -p data/raw/mmdec data/processed data/interim checkpoints outputs tables figures logs

# A. Authentic UEIKAP SAR + Human Polygon Masks
if not os.path.exists('data/raw/UEIKAP_SAR.zip'):
    !curl -L -C - "https://zenodo.org/records/18391027/files/UEIKAP_SAR.zip?download=1" -o data/raw/UEIKAP_SAR.zip
    !unzip -q -o data/raw/UEIKAP_SAR.zip -d data/raw/UEIKAP_SAR

raw_dir = 'data/raw'
tgt_dir = os.path.join(raw_dir, 'UEIKAP_SAR')
for root, dirs, files in os.walk(raw_dir):
    if 'images' in dirs and root != tgt_dir:
        for d in dirs:
            src_p = os.path.join(root, d)
            dst_p = os.path.join(tgt_dir, d)
            if not os.path.exists(dst_p):
                shutil.move(src_p, dst_p)

# Build/refresh UEIKAP segmentation images
!python -m src.data.ueikap_parser
!python -m src.data.build_master_dataset
!python -m src.wake.package_segmentation_data

# B. MMDEC AIS Telemetry (430 MB) + Corners (5 MB) + SAR Chunks Zip (10.5 GB, streamed directly!)
mmdec_dir = 'data/raw/mmdec'
if not os.path.exists(f'{mmdec_dir}/Dataset_AIS_POS.parquet'):
    !curl -L -C - "https://zenodo.org/api/records/17491518/files/Dataset_AIS_POS.parquet/content" -o {mmdec_dir}/Dataset_AIS_POS.parquet

if not os.path.exists(f'{mmdec_dir}/Dataset_S1_chunks_corners.zip'):
    !curl -L -C - "https://zenodo.org/api/records/17491518/files/Dataset_S1_chunks_corners.zip/content" -o {mmdec_dir}/Dataset_S1_chunks_corners.zip

if not os.path.exists(f'{mmdec_dir}/Dataset_S1_chunks.zip'):
    print("Downloading MMDEC Dataset_S1_chunks.zip (~10.5 GB, kept compressed and streamed in-memory)...")
    !curl -L -C - "https://zenodo.org/api/records/17491518/files/Dataset_S1_chunks.zip/content" -o {mmdec_dir}/Dataset_S1_chunks.zip

print("All datasets verified and ready!")

In [ ]:
print("=== 4. STAGE 1: TRAINING HYBRID WAKE-NET (ResNet-50 + scSE + Radon) ===")
!python -m src.wake.train_wake_net --epochs 35 --batch_size 8 --lr 3e-4

In [ ]:
print("=== 5. STAGE 2: MMDEC HARVESTER + WAKE-NET AUTO-MASKING + RADON GEOMETRY ===")
# Use --max_straight -1 --max_turning -1 to harvest ALL qualified cruising & turning ships!
# Default --max_straight 4000 --max_turning -1 harvests up to 4,000 straight + ALL turning ships.
!python -m src.data.mmdec_extract_and_mask --min_sog 3.5 --min_turn_rot 1.5 --max_straight 4000 --max_turning -1 --batch_size 32

In [ ]:
print("=== 6. STAGE 3: TRAINING DUAL-BRANCH CONVNEXT + LEE + GEOMETRY ENSEMBLE ===")
!python kaggle/KAGGLE_MASTER_EXPANDED_SUITE.py --epochs 25 --batch_size 32 --lr 2e-4

In [ ]:
print("=== 7. PACKAGING WEIGHTS, METRICS & VISUALIZATIONS ===")
!zip -r /kaggle/working/master_thesis_results_3k.zip checkpoints/ logs/ tables/ figures/ outputs/ data/processed/*.parquet data/processed/*.json data/processed/*.csv

print("\n" + "=" * 75)
print("SUCCESS! Complete Master Thesis Pipeline Finished.")
print("Download '/kaggle/working/master_thesis_results_3k.zip' from Kaggle Output!")
print("=" * 75)